# Select Transfer Function


In [ ]:
import hickle as hkl

from lib.utils import (
    G_sp_to_ctrl,
    input_names,
    output_labels,
    output_names,
)

tf = (3, 2)  # G_{4,3}: xA1_FR

G_matrix = hkl.load("../outputs/G_simplified.hkl")
G_name = f"{output_names[tf[0]]}_{input_names[tf[1]]}"
G_ylabel = f"{output_labels[tf[0]]}"  # / {output_units[tf[0]]}"
print(G_name)

G = G_sp_to_ctrl(G_matrix[tf])


xA1_FR


# Plot FFT


In [2]:
import control as ctrl
import numpy as np

from lib.plots import plot_fft

_t, _y = ctrl.step_response(G)
t, y = np.array(_t), np.array(_y)


plot_fft(t, y, f"discrete/{G_name}_fft")


Plot saved to ../figures/discrete/xA1_FR_fft.png


# Discretization


In [3]:
ws = 150  # frequência do sistema (rad/h)
wc = ws * 2  # frequência de amostragem (rad/h)
Ts = 2 * np.pi / wc  # período de amostragem (h)

print("Pelo gráfico de FFT,", end=" ")
print(f"a frequência do sistema (ws) é aproximadamente {ws} rad/h.")
print("Usando o Teorema de Amostragem de Shannon,", end=" ")
print("a frequência de amostragem deve ser pelo menos o dobro da frequência de corte.")
print(f"Ou seja, {wc} rad/h.")
print(f"Isso corresponde a um período de amostragem de aproximadamente {Ts:.4f} h.")
print("Usando a regra prática de 50 pontos para a dinamica (que dura 2h),", end=" ")
print(f"o período de amostragem seria {2 / 50:.4f} h.")
print("Com esses dois critérios, escolhemos o período de amostragem menor:", end=" ")
print(f"{Ts:.4f} h.")

td = np.arange(t[0], t[-1] + Ts, Ts)
yd = np.interp(td, t, y)


Pelo gráfico de FFT, a frequência do sistema (ws) é aproximadamente 150 rad/h.
Usando o Teorema de Amostragem de Shannon, a frequência de amostragem deve ser pelo menos o dobro da frequência de corte.
Ou seja, 300 rad/h.
Isso corresponde a um período de amostragem de aproximadamente 0.0209 h.
Usando a regra prática de 50 pontos para a dinamica (que dura 2h), o período de amostragem seria 0.0400 h.
Com esses dois critérios, escolhemos o período de amostragem menor: 0.0209 h.


In [4]:
from lib.plots import plot_or_show, plt

plt.figure(figsize=(8, 4))

# sinal contínuo
plt.plot(t, y, label="Sinal continuo")

# amostras discretas (apenas pontos vermelhos)
plt.scatter(td, yd, color="red", s=20, zorder=3, label="Sinal amostrado")

plt.xlabel("Tempo / h")
plt.ylabel(G_ylabel)
plt.grid(True)
plt.legend()

plot_or_show("discrete/amostragem")


Plot saved to ../figures/discrete/amostragem.png


# Polos discretos


In [5]:
poles_s = ctrl.poles(G)

poles_z = np.exp(poles_s * Ts)

print("Polos contínuos:")
print(poles_s)

print("\nPolos discretos:")
print(poles_z)


fig, ax = plt.subplots(figsize=(6, 6))

# Círculo unitário
theta = np.linspace(0, 2 * np.pi, 500)

ax.fill(np.cos(theta), np.sin(theta), alpha=0.15, color="tab:blue")

ax.plot(np.cos(theta), np.sin(theta), color="tab:blue", label="Região estável")

# Polos
ax.scatter(
    np.real(poles_z),
    np.imag(poles_z),
    marker="x",
    s=100,
    color="red",
    label="Polos discretos",
)

# Eixos
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)

ax.set_xlabel("Parte Real")
ax.set_ylabel("Parte Imaginária")
ax.set_aspect("equal")

ax.legend(loc="upper right")
plot_or_show("discrete/pzplot")


Polos contínuos:
[-52.87124085+19.92478864j -52.87124085-19.92478864j
 -48.03750216+19.6198806j  -48.03750216-19.6198806j
  -6.96029005 +0.j          -3.17939438 +0.j        ]

Polos discretos:
[0.30208254+0.13392604j 0.30208254-0.13392604j 0.33520582+0.14605683j
 0.33520582-0.14605683j 0.86435132+0.j         0.93557957+0.j        ]
Plot saved to ../figures/discrete/pzplot.png
